In [9]:
# ============================================================
# TAHAP 1: Konversi PDF → TXT (Case Base Building)
# Domain: Pidana Khusus Narkotika & Psikotropika
# ============================================================

import os
import re
import json
import logging
import pandas as pd
from tqdm import tqdm
from pdfminer.high_level import extract_text
from pdfminer.pdfparser import PDFSyntaxError

# Setup logging
os.makedirs('../logs', exist_ok=True)
logging.basicConfig(
    filename='../logs/cleaning.log',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

# Path folder
PDF_DIR = '../data/raw/pdf/'
TXT_DIR = '../data/raw/txt/'
os.makedirs(PDF_DIR, exist_ok=True)
os.makedirs(TXT_DIR, exist_ok=True)

print(f'PDF folder : {PDF_DIR}')
print(f'TXT folder : {TXT_DIR}')
print(f'Jumlah PDF : {len([f for f in os.listdir(PDF_DIR) if f.endswith(".pdf")])}')

PDF folder : ../data/raw/pdf/
TXT folder : ../data/raw/txt/
Jumlah PDF : 48


In [10]:
# ============================================================
# Fungsi Konversi PDF → TXT + Cleaning
# ============================================================

def extract_text_from_pdf(pdf_path):
    """Ekstrak teks dari file PDF menggunakan pdfminer"""
    try:
        text = extract_text(pdf_path)
        return text
    except PDFSyntaxError as e:
        logging.error(f'PDFSyntaxError pada {pdf_path}: {e}')
        return None
    except Exception as e:
        logging.error(f'Error pada {pdf_path}: {e}')
        return None

def clean_text(text):
    """
    Membersihkan teks putusan:
    1. Hapus karakter tidak perlu
    2. Normalisasi spasi
    3. Lowercase
    4. Hapus nomor halaman
    """
    if not text:
        return None

    # Hapus karakter non-ASCII yang tidak perlu
    text = re.sub(r'[^\x00-\x7F\u00C0-\u024F\u1E00-\u1EFF]', ' ', text)

    # Hapus nomor halaman (pola: "Halaman X dari Y" atau angka berdiri sendiri)
    text = re.sub(r'Halaman\s+\d+\s+dari\s+\d+', '', text, flags=re.IGNORECASE)
    text = re.sub(r'hal\.\s*\d+\s*/\s*\d+', '', text, flags=re.IGNORECASE)

    # Hapus header/footer umum putusan MA
    text = re.sub(r'Mahkamah Agung Republik Indonesia', '', text, flags=re.IGNORECASE)
    text = re.sub(r'Direktori Putusan', '', text, flags=re.IGNORECASE)
    text = re.sub(r'putusan\.mahkamahagung\.go\.id', '', text, flags=re.IGNORECASE)
    text = re.sub(r'Disclaimer.*?(?=\n)', '', text, flags=re.IGNORECASE)

    # Normalisasi spasi dan newline berlebihan
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r' {2,}', ' ', text)
    text = text.strip()

    return text

def validate_text(text, min_ratio=0.8, min_chars=1000):
    """
    Validasi keutuhan teks:
    - Minimal 1000 karakter
    - Minimal 80% berupa huruf/angka (bukan noise)
    """
    if not text or len(text) < min_chars:
        return False
    alpha_count = sum(1 for c in text if c.isalnum() or c.isspace())
    ratio = alpha_count / len(text)
    return ratio >= min_ratio

# Proses semua PDF
pdf_files = [f for f in os.listdir(PDF_DIR) if f.endswith('.pdf')]
print(f'\nMemproses {len(pdf_files)} file PDF...\n')

success_count = 0
fail_count    = 0
results       = []

for i, pdf_file in enumerate(tqdm(pdf_files, desc='Konversi PDF→TXT')):
    pdf_path = os.path.join(PDF_DIR, pdf_file)
    txt_name = pdf_file.replace('.pdf', '.txt')
    txt_path = os.path.join(TXT_DIR, txt_name)
    case_id  = f'case_{str(i+1).zfill(3)}'

    # Ekstrak teks
    raw_text   = extract_text_from_pdf(pdf_path)
    clean      = clean_text(raw_text)
    is_valid   = validate_text(clean) if clean else False

    if is_valid:
        # Simpan ke .txt
        with open(txt_path, 'w', encoding='utf-8') as f:
            f.write(clean)
        success_count += 1
        logging.info(f'SUCCESS: {pdf_file} → {txt_name} ({len(clean)} chars)')
        results.append({'case_id': case_id, 'pdf_file': pdf_file,
                        'txt_file': txt_name, 'status': 'success',
                        'char_count': len(clean)})
    else:
        fail_count += 1
        logging.warning(f'FAILED: {pdf_file} — teks tidak valid atau terlalu pendek')
        results.append({'case_id': case_id, 'pdf_file': pdf_file,
                        'txt_file': None, 'status': 'failed', 'char_count': 0})

print(f'\n✅ Berhasil : {success_count} file')
print(f'❌ Gagal    : {fail_count} file')
print(f'📄 Log tersimpan di: ../logs/cleaning.log')

df_log = pd.DataFrame(results)
print(df_log)


Memproses 48 file PDF...



Konversi PDF→TXT: 100%|██████████| 48/48 [00:57<00:00,  1.20s/it]


✅ Berhasil : 48 file
❌ Gagal    : 0 file
📄 Log tersimpan di: ../logs/cleaning.log
     case_id                                    pdf_file  \
0   case_001    5_pid.sus_2026_pn_bjm_20260615001800.pdf   
1   case_002  707_pid.sus_2025_pn_bjm_20260615183740.pdf   
2   case_003  708_pid.sus_2025_pn_bjm_20260615184004.pdf   
3   case_004  711_pid.sus_2025_pn_bjm_20260615224549.pdf   
4   case_005  732_pid.sus_2025_pn_bjm_20260615224610.pdf   
5   case_006  733_pid.sus_2025_pn_bjm_20260615184204.pdf   
6   case_007  741_pid.sus_2025_pn_bjm_20260615225122.pdf   
7   case_008  743_pid.sus_2025_pn_bjm_20260615185826.pdf   
8   case_009  743_pid.sus_2025_pn_bjm_20260615185838.pdf   
9   case_010  747_pid.sus_2025_pn_bjm_20260615183552.pdf   
10  case_011  748_pid.sus_2025_pn_bjm_20260615182947.pdf   
11  case_012  749_pid.sus_2025_pn_bjm_20260615224511.pdf   
12  case_013  750_pid.sus_2025_pn_bjm_20260615224451.pdf   
13  case_014  751_pid.sus_2025_pn_bjm_20260615183631.pdf   
14  case_015  755

In [11]:
# ============================================================
# TAHAP 2: Case Representation — Ekstrak Metadata
# ============================================================

def extract_metadata(text, case_id, filename):
    """
    Ekstrak metadata penting dari teks putusan narkotika:
    - Nomor perkara
    - Tanggal putusan
    - Jenis perkara
    - Pasal yang digunakan
    - Nama terdakwa
    - Amar putusan
    - Dakwaan
    """
    metadata = {
        'case_id'        : case_id,
        'filename'       : filename,
        'no_perkara'     : None,
        'tanggal'        : None,
        'jenis_perkara'  : 'Pidana Khusus Narkotika & Psikotropika',
        'terdakwa'       : None,
        'pasal'          : None,
        'dakwaan'        : None,
        'amar_putusan'   : None,
        'ringkasan_fakta': None,
        'word_count'     : len(text.split()) if text else 0,
        'text_full'      : text[:5000] if text else None  # simpan 5000 char pertama
    }

    if not text:
        return metadata

    # ── Nomor Perkara ──────────────────────────────────────
    pola_perkara = [
        r'Nomor\s*[:\s]*(\d+/Pid\.Sus[^\n\r]*)',
        r'(\d+\s*/\s*Pid\.Sus[^\n\r]*)',
        r'PUTUSAN\s+Nomor\s*[:\s]*([^\n\r]+)',
        r'No\.\s*(\d+/[^\n\r]+)',
    ]
    for pola in pola_perkara:
        match = re.search(pola, text, re.IGNORECASE)
        if match:
            metadata['no_perkara'] = match.group(1).strip()[:100]
            break

    # ── Tanggal Putusan ────────────────────────────────────
    pola_tgl = [
        r'(?:tanggal|pada)\s+(\d{1,2}\s+\w+\s+\d{4})',
        r'(\d{1,2}\s+(?:Januari|Februari|Maret|April|Mei|Juni|Juli|'
        r'Agustus|September|Oktober|November|Desember)\s+\d{4})',
    ]
    for pola in pola_tgl:
        match = re.search(pola, text, re.IGNORECASE)
        if match:
            metadata['tanggal'] = match.group(1).strip()
            break

    # ── Nama Terdakwa ──────────────────────────────────────
    pola_tdkw = [
        r'(?:terdakwa|nama lengkap)\s*[:\s]+([A-Z][A-Za-z\s]+?)(?:\n|;|,)',
        r'(?:Terdakwa\s+bernama)\s+([A-Z][A-Za-z\s]+?)(?:\n|;|,)',
    ]
    for pola in pola_tdkw:
        match = re.search(pola, text, re.IGNORECASE)
        if match:
            metadata['terdakwa'] = match.group(1).strip()[:100]
            break

    # ── Pasal yang Digunakan ───────────────────────────────
    pola_pasal = [
        r'(Pasal\s+\d+[^\n\r]*(?:Undang-Undang|UU)[^\n\r]*Narkotika[^\n\r]*)',
        r'(Pasal\s+\d+\s+(?:ayat\s*\(\d+\)\s*)?(?:jo\.?\s*Pasal\s+\d+\s*)?'
        r'(?:Undang-Undang|UU)[^\n\r]*)',
        r'(Pasal\s+\d+[^\n\r]{0,80})',
    ]
    pasals = []
    for pola in pola_pasal:
        matches = re.findall(pola, text, re.IGNORECASE)
        pasals.extend(matches[:3])
    if pasals:
        metadata['pasal'] = ' | '.join(list(dict.fromkeys(pasals)))[:300]

    # ── Dakwaan ────────────────────────────────────────────
    match = re.search(
        r'(?:DAKWAAN|Surat Dakwaan)[:\s]*(.*?)(?=TUNTUTAN|FAKTA|PERTIMBANGAN|$)',
        text, re.IGNORECASE | re.DOTALL)
    if match:
        metadata['dakwaan'] = match.group(1).strip()[:500]

    # ── Amar Putusan ───────────────────────────────────────
    match = re.search(
        r'(?:MENGADILI|AMAR PUTUSAN)[:\s]*(.*?)(?=Demikian|ditetapkan|$)',
        text, re.IGNORECASE | re.DOTALL)
    if match:
        metadata['amar_putusan'] = match.group(1).strip()[:500]

    # ── Ringkasan Fakta ────────────────────────────────────
    match = re.search(
        r'(?:FAKTA HUKUM|FAKTA PERSIDANGAN|barang bukti)[:\s]*(.*?)(?=PERTIMBANGAN|DAKWAAN|$)',
        text, re.IGNORECASE | re.DOTALL)
    if match:
        metadata['ringkasan_fakta'] = match.group(1).strip()[:500]
    else:
        # Fallback: ambil paragraf pertama yang cukup panjang
        paragraphs = [p.strip() for p in text.split('\n\n') if len(p.strip()) > 100]
        if paragraphs:
            metadata['ringkasan_fakta'] = paragraphs[0][:500]

    return metadata

In [12]:
# ============================================================
# Proses semua file TXT → Ekstrak metadata → Simpan CSV
# ============================================================

os.makedirs('../data/processed', exist_ok=True)

txt_files   = [f for f in os.listdir(TXT_DIR) if f.endswith('.txt')]
all_metadata = []

print(f'Mengekstrak metadata dari {len(txt_files)} file TXT...\n')

for i, txt_file in enumerate(tqdm(txt_files, desc='Ekstrak Metadata')):
    txt_path = os.path.join(TXT_DIR, txt_file)
    case_id  = f'case_{str(i+1).zfill(3)}'

    with open(txt_path, 'r', encoding='utf-8') as f:
        text = f.read()

    metadata = extract_metadata(text, case_id, txt_file)
    all_metadata.append(metadata)
    logging.info(f'Metadata extracted: {txt_file}')

# Buat DataFrame
df = pd.DataFrame(all_metadata)

# Simpan ke CSV
# Simpan ke CSV
csv_path = '../data/processed/cases.csv'
df.to_csv(csv_path, index=False, encoding='utf-8-sig')

# Simpan ke JSON — PERBAIKAN: cek dulu kolom yang ada
json_path = '../data/processed/cases.json'
cols_to_drop = [c for c in ['text_full'] if c in df.columns]  # ← PERBAIKAN
df.drop(columns=cols_to_drop).to_json(
    json_path, orient='records',
    force_ascii=False, indent=2
)

print(f'\n✅ Dataset berhasil disimpan!')
print(f'   CSV  : {csv_path}')
print(f'   JSON : {json_path}')
print(f'   Total kasus: {len(df)}')
print(f'\n📊 Preview data:')
print(df[['case_id','no_perkara','tanggal','terdakwa','pasal','word_count']].head(10))

Mengekstrak metadata dari 48 file TXT...



Ekstrak Metadata: 100%|██████████| 48/48 [00:00<00:00, 69.62it/s]


✅ Dataset berhasil disimpan!
   CSV  : ../data/processed/cases.csv
   JSON : ../data/processed/cases.json
   Total kasus: 48

📊 Preview data:
    case_id                                         no_perkara  \
0  case_001  5/Pid.Sus/2026/PN BjmDEMI KEADILAN BERDASARKAN...   
1  case_002  707/Pid.Sus/2025/PN BjmDEMI KEADILAN BERDASARK...   
2  case_003  708/Pid.Sus/2025/PN BjmDEMI KEADILAN BERDASARK...   
3  case_004  711/Pid.Sus/2025/PN BjmDEMI KEADILAN BERDASARK...   
4  case_005  732/Pid.Sus/2025/PN BjmDEMI KEADILAN BERDASARK...   
5  case_006  733/Pid.Sus/2025/PN BjmDEMI KEADILAN BERDASARK...   
6  case_007  741/Pid.Sus/2025/PN BjmDEMI KEADILAN BERDASARK...   
7  case_008  743/Pid.Sus/2025/PN BjmDEMI KEADILAN BERDASARK...   
8  case_009  743/Pid.Sus/2025/PN BjmDEMI KEADILAN BERDASARK...   
9  case_010  747/Pid.Sus/2025/PN BjmDEMI KEADILAN BERDASARK...   

           tanggal                                           terdakwa  \
0  28 Agustus 2025                      didampingi Advoka

In [13]:
# ============================================================
# Statistik & Validasi Dataset
# ============================================================

print('='*60)
print('STATISTIK DATASET')
print('='*60)
print(f'Total kasus          : {len(df)}')
print(f'Kasus dengan no_perkara : {df["no_perkara"].notna().sum()}')
print(f'Kasus dengan tanggal    : {df["tanggal"].notna().sum()}')
print(f'Kasus dengan terdakwa   : {df["terdakwa"].notna().sum()}')
print(f'Kasus dengan pasal      : {df["pasal"].notna().sum()}')
print(f'Kasus dengan dakwaan    : {df["dakwaan"].notna().sum()}')
print(f'Kasus dengan amar       : {df["amar_putusan"].notna().sum()}')
print(f'\nRata-rata word count : {df["word_count"].mean():.0f} kata')
print(f'Min word count       : {df["word_count"].min()} kata')
print(f'Max word count       : {df["word_count"].max()} kata')

print('\n📌 Contoh no_perkara:')
print(df['no_perkara'].dropna().head(5).tolist())

print('\n📌 Contoh pasal:')
print(df['pasal'].dropna().head(3).tolist())

STATISTIK DATASET
Total kasus          : 48
Kasus dengan no_perkara : 48
Kasus dengan tanggal    : 48
Kasus dengan terdakwa   : 48
Kasus dengan pasal      : 48
Kasus dengan dakwaan    : 48
Kasus dengan amar       : 48

Rata-rata word count : 10014 kata
Min word count       : 5394 kata
Max word count       : 20973 kata

📌 Contoh no_perkara:
['5/Pid.Sus/2026/PN BjmDEMI KEADILAN BERDASARKAN KETUHANAN YANG MAHA ESAPengadilan Negeri Banjarmasin ', '707/Pid.Sus/2025/PN BjmDEMI KEADILAN BERDASARKAN KETUHANAN YANG MAHA ESAPengadilan Negeri Banjarmasi', '708/Pid.Sus/2025/PN BjmDEMI KEADILAN BERDASARKAN KETUHANAN YANG MAHA ESAPengadilan Negeri Banjarmasi', '711/Pid.Sus/2025/PN BjmDEMI KEADILAN BERDASARKAN KETUHANAN YANG MAHA ESAPengadilan Negeri Banjarmasi', '732/Pid.Sus/2025/PN BjmDEMI KEADILAN BERDASARKAN KETUHANAN YANG MAHA ESAPengadilan Negeri Banjarmasi']

📌 Contoh pasal:
['Pasal 114 ayat (2) Undang-Undang Republik Indonesia Nomor 35 Tahun2009 Tentang Narkotika jo Pasal 622 ayat (1) huruf w